# React — Context

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> This topic is playground work. Context is about how data moves through a component tree, and
> a notebook cell has no tree.

## LESSON 55 — Prop drilling

Before the solution, the problem — properly felt, because Context is easy to reach for and easy
to overuse.

### The situation

LESSON 29 taught you to lift state to the closest common parent. That is the right answer, and
it has a cost that grows with the distance:

> Passing props can become verbose and inconvenient when you need to pass some prop deeply
> through the tree, or if many components need the same prop. The nearest common ancestor could
> be far removed from the components that need data, and lifting state up that high can lead to
> a situation called **"prop drilling"**.

In the playground experiment, `user` is needed in exactly **one** place — the avatar, five
levels down. Count what that costs:

```text
   App — owns the user
   Layout — only passes user on
   Page — only passes user on
   Header — only passes user on
   Toolbar — only passes user on
   Avatar — USES user: Ada Lovelace
```

Five components mention `user`. One uses it. The other four have a prop in their signature, a
prop in their JSX, and no reason to care.

### Why it is worth taking seriously

It is not about typing. Each of those four components now:

- **cannot be moved** without rethreading the prop,
- **cannot be understood alone** — reading `Header` tells you it has a user, but not why,
- **must be edited** when the shape of `user` changes, even though it never reads a field,
- and becomes a **plausible place to look** when something about the user is wrong, which
  wastes time during debugging.

Multiply by three or four such values — the current user, the theme, the locale, a `dispatch`
from topic 17 — and the middle of your app becomes plumbing.

### But do not reach for Context yet

React is unusually direct about this, and it comes *before* the solution for a reason:

> Just because you need to pass some props several levels deep doesn't mean you should put that
> information into context.

Two alternatives to try first.

**1. Just pass the props.**

> If your components are not trivial, it's not unusual to pass a dozen props down through a
> dozen components. It may feel like a slog, but it makes it very clear which components use
> which data! The person maintaining your code will be glad you've made the data flow explicit
> with props.

Explicit is a feature. A prop you can trace beats a value that arrives from nowhere.

**2. Extract components and pass JSX as `children`.**

> If you pass some data through many layers of intermediate components that don't use that data
> (and only pass it further down), this often means that you forgot to extract some components
> along the way. For example, maybe you pass data props like `posts` to visual components that
> don't use them directly, like `<Layout posts={posts} />`. Instead, make `Layout` take
> `children` as a prop, and render `<Layout><Posts posts={posts} /></Layout>`. This reduces the
> number of layers between the component specifying the data and the one that needs it.

This one is worth dwelling on, because it is the option people forget. `children` is LESSON 5 —
you have had it since the second topic — and it removes layers rather than routing around them.
If `Layout` takes `children`, it no longer sits between the data and its user at all.

> If neither of these approaches works well for you, consider context.

That sentence is the actual entry condition for LESSON 56.

### Key Notes

- **Prop drilling** is passing a prop through components that do not use it, only to reach one
  that does.
- The cost is coupling and confusion, not keystrokes: those components cannot be moved, read or
  changed independently.
- Try passing props first — explicit data flow is a feature, not a failure.
- Try `children` second. Layers that only forward data often should not exist.
- Context is what you reach for when neither works.

### Example

**In the playground.** No cell — the problem is a shape in a component tree, and a notebook has
no tree to shape.

Point `playground/src/App.jsx` at `./experiments/25-drilling.jsx` and open the console. Read
the six lines and note which one says "USES". Then open the file and count how many times the
word `user` appears.

### Exercise

**In the playground**, in `25-drilling.jsx`.

1. Count it precisely: how many components take `user` as a prop, and how many read a field
   from it? Write both numbers in a comment.
2. Add a second drilled value — a `theme` string that only `Avatar` uses — threading it through
   the same five components. How many lines did you have to touch to deliver one string?
3. **Apply React's second alternative.** Rewrite `Layout`, `Page` and `Header` to take
   `children` instead of `user`, and render the tree from `Experiment25` so that `Toolbar`
   receives `user` directly. How many components mention `user` now?
4. In a comment: the `children` version is shorter, and it is not free. What did you give up,
   and when would you not want to do it?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

For each, say whether you would **pass props**, use **`children`**, or that it is a genuine
case for **Context** — with one line of reasoning.

1. The signed-in user, needed by a header avatar, a comment form and a permissions check on
   three different screens.
2. A `posts` array passed through `Layout` → `Sidebar` → `PostList`, where only `PostList`
   uses it.
3. The current colour theme, read by roughly thirty components at every depth.
4. A `productId` passed from a page component to the three sections that make up that page.
5. A `dispatch` function from topic 17, needed by a dozen components across a large form.
6. A `selectedRow` used by a table and the detail panel directly beside it.

Then answer: two of these look identical in shape — a value threaded through several layers —
and get different answers. Which two, and what distinguishes them?

In [ ]:
// Your code here

## LESSON 56 — `createContext`, providing, and reading

You have tried passing props and you have tried `children`, and the value is still travelling
through components that do not want it. Now Context earns its place.

### Three steps

React's own list:

> 1. **Create** a context.
> 2. **Use** that context from the component that needs the data.
> 3. **Provide** that context from the component that specifies the data.

**Create** — outside any component, usually in its own file:

```jsx
import { createContext } from "react";

export const UserContext = createContext(null);
```

The argument is the **default value**, used only when a component reads the context with no
provider above it. `null` is a fine default when there is no sensible fallback — and reading a
`null` user will fail loudly, which is usually what you want.

**Provide** — wrap the subtree that should see the value:

```jsx
<UserContext value={user}>
  <Layout />
</UserContext>
```

**Use** — read it, at any depth, with no props in between:

```jsx
import { useContext } from "react";

function Avatar() {
  const user = useContext(UserContext);
  return <span>{user.name}</span>;
}
```

`useContext` is a Hook, so LESSON 37 applies: top level of the component, never in a condition.

### The modern provider syntax

In React 19 the context **is** the provider:

```jsx
<UserContext value={user}>      {/* React 19 */}
<UserContext.Provider value={user}>   {/* older, still works */}
```

Both work today — measured in the playground, both render identically. But React's own
statement is clear about direction:

> New Context providers can use `<Context>`… In future versions we will deprecate
> `<Context.Provider>`.

So: write `<UserContext value={…}>` in new code, and recognise `.Provider` when you read
anything written before React 19. That is the same arrangement as `forwardRef` in LESSON 49 —
one form to write, one form to recognise.

### What it actually changes

Compare experiment 25 with experiment 26. Same tree, same avatar, same data. The difference:

```text
   Layout — knows nothing about the user
   Page — knows nothing about the user
   Header — knows nothing about the user
   Toolbar — knows nothing about the user
   Avatar — reads user "Ada Lovelace" and theme "light"
```

> You can insert as many components as you like between the component that provides context and
> the one that uses it. This includes both built-in components like `<div>` and components you
> might build yourself.

> Context passes through any components in the middle.

Four components got shorter and stopped knowing things they had no business knowing.

### It is not static

> Context is not limited to static values. If you pass a different value on the next render,
> React will update all the components reading it below! This is why context is often used in
> combination with state.

The provider's value is usually state. Measured: clicking **switch user** in the experiment
changes the avatar five levels down, with no prop anywhere in between.

### Nested providers override

A provider applies to its own subtree, and a nearer one wins:

```jsx
<UserContext value={ada}>
  <Avatar />                    {/* Ada */}
  <UserContext value={katherine}>
    <Avatar />                  {/* Katherine */}
  </UserContext>
</UserContext>
```

Measured in the experiment: the outer avatar reads `Ada Lovelace (AL)`, the nested one reads
`Katherine Johnson (KJ)`, and switching the outer user changes the first while the second stays
put. Each component reads the **nearest** provider above it.

### Key Notes

- Three steps: **create** the context, **provide** a value, **use** it where it is needed.
- Write `<MyContext value={…}>` — the React 19 form. `.Provider` is for reading older code.
- Components in between need no props and no knowledge; context passes straight through them.
- The value is usually state, so changing it re-renders every reader below.
- The nearest provider above a component wins.

### Example

**In the playground.** No cell — Context is about a tree.

Point `playground/src/App.jsx` at `./experiments/26-context.jsx`, then open
`25-drilling.jsx` beside it. The two files render the same avatar. Read the middle components in
each and note what they know.

### Exercise

**In the playground**, in `26-context.jsx`.

1. With the console open, click **switch user**. Which components re-rendered, and which of
   them mention `user` in their code? Explain the gap in one sentence.
2. The nested provider shows a different name. Change the outer user and watch what the nested
   avatar does. What rule does that demonstrate?
3. Delete the `<ThemeContext value={theme}>` wrapper around the nested block, so the nested
   `Avatar` has no theme provider above it. What theme does it use, and where did that value
   come from?
4. Change `createContext(null)` to `createContext({ name: "Guest", initials: "G" })` and then
   render an `<Avatar />` **outside** both providers. What appears, and when would a default
   like that be a good idea rather than a way to hide a bug?
5. Swap one provider to the older `<UserContext.Provider value={…}>` form. Does anything change
   on screen or in the console? What does that tell you about which one to write?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

```jsx
// A
function Avatar() {
  if (!isLoggedIn) return null;
  const user = useContext(UserContext);
  return <span>{user.name}</span>;
}

// B
const ThemeContext = createContext("light");
function App() {
  const [theme, setTheme] = useState("dark");
  return <Layout />;                       // no provider anywhere
}

// C
function App() {
  const [user, setUser] = useState(ada);
  return (
    <UserContext value={{ user, setUser }}>
      <Layout />
    </UserContext>
  );
}
```

1. **A** breaks a rule from an earlier lesson. Which, and what is the symptom?
2. **B** renders without crashing and shows the wrong theme forever. Explain exactly why, and
   say what is missing.
3. **C** is correct and contains a performance trap that LESSON 57 is about. Predict what it is
   before you read on.

In [ ]:
// Your code here

## LESSON 57 — One provider per concern, and when not to use Context

Context works. This lesson is about using it well, and about not using it.

### One provider per concern

The experiment provides two contexts, not one object containing both:

```jsx
<UserContext value={user}>
  <ThemeContext value={theme}>
    <Layout />
  </ThemeContext>
</UserContext>
```

rather than:

```jsx
<AppContext value={{ user, theme, locale, dispatch }}>
```

The nesting looks heavier and is the better arrangement, for a reason you can state precisely:
**every component reading a context re-renders when that context's value changes.** One
combined context means a component that only cares about the theme re-renders when the user
changes, and vice versa. Splitting them means each reader only hears about what it asked for.

It also reads better. `useContext(ThemeContext)` says what the component depends on;
`useContext(AppContext).theme` says the component depends on everything and happens to use one
field.

### The value-identity trap

This is the thing to get right, and mini challenge C in LESSON 56 pointed at it:

```jsx
function App() {
  const [user, setUser] = useState(ada);

  return (
    <UserContext value={{ user, setUser }}>   {/* a NEW object every render */}
      <Layout />
    </UserContext>
  );
}
```

`{ user, setUser }` is an object literal built during render, so it is a different value every
time `App` renders — LESSON 40's identity rule, in a third place. React compares context values
the same way it compares everything else, so **every reader re-renders whenever `App` renders**,
even if `user` itself never changed.

Two honest responses:

**1. Do not combine unless you need to.** Provide `user` and `setUser` as two contexts, or
provide just the value most components need. Half of this problem is self-inflicted.

**2. If you must combine, the fix is memoisation — and that is topic 23.** There is a Hook for
keeping an object's identity stable between renders, you have not been taught it yet, and you
should not reach for it before you can measure whether it matters. This course teaches
measurement first, on purpose.

For now the useful thing is to **recognise** the shape: an object or array literal as a context
value means every reader re-renders on every provider render. Whether that costs anything
depends on how many readers there are and how expensive they are — which is a question for the
Profiler (LESSON 35), not for a reflex.

### When not to use Context

React's warning is short and deserved:

> Context is very tempting to use! However, this also means it's too easy to overuse it.

Three situations where it is the wrong tool:

**It is not a state manager.** Context *transports* a value; it does not own it, change it or
optimise it. The state still lives in a component, with `useState` or `useReducer`. "I'll put
it in Context" is not an answer to "where should this state live" (LESSON 29).

**It is not for two components that sit beside each other.** If the distance is one or two
levels, props are clearer and cheaper. Context earns its keep across distance, not across a
gap.

**It is not a way to avoid thinking about data flow.** A value that arrives from nowhere is
harder to trace than one passed explicitly, and every reader becomes coupled to a provider
existing somewhere above it. That is a real trade, worth making deliberately.

### What it is genuinely good for

React's list is a good one, and every item shares a shape — **many readers, at many depths, that
rarely changes**:

- **Theming** — the appearance of the whole app.
- **Current account** — who is signed in.
- **Routing** — "most routing solutions use context internally to hold the current route. This
  is how every link 'knows' whether it's active or not." You will use one in topic 20.
- **Managing state** — "It is common to use a reducer together with context to manage complex
  state and pass it down to distant components without too much hassle."

That last one is your topic 17 reducer meeting your topic 18 context, and it is exactly what
Mini-project 3 asks for.

### Key Notes

- **One provider per concern.** Combined contexts re-render readers that did not care.
- A **new object literal as a context value** re-renders every reader on every provider render
  (LESSON 40's identity rule again). Recognise it; the fix is topic 23's.
- Context **transports** state, it does not own or manage it.
- It pays off across distance with many readers — theme, current user, routing, a shared
  reducer. For one or two levels, use props.

### Example

**In the playground.** No cell — this lesson is about re-render behaviour across a tree.

In `26-context.jsx`, note that `user` and `theme` are two separate contexts. Toggle the theme
and watch which components log a render; then switch the user and watch again.

### Exercise

**In the playground**, in `26-context.jsx`.

1. Combine the two contexts into one `AppContext` carrying `{ user, theme }`, and update
   `Avatar` to read both from it. Toggle the theme and switch the user. What changed about which
   components re-render? Use the Profiler (LESSON 35) if the console is not clear enough.
2. Put the two contexts back. Now change the provider to `<UserContext value={{ user }}>` — an
   object literal. Click **toggle theme** (which does *not* change the user) and watch whether
   `Avatar` re-renders. Explain what you see using LESSON 40's rule.
3. Add a `dispatch` from a `useReducer` to the tree via a **second** context, so a deep
   component can dispatch without a prop. This is the topic 17 + topic 18 combination the
   lesson names; keep the state and the dispatch in separate contexts and say in a comment why.
4. In a comment: name one value in this experiment that should **not** be in Context at all,
   and say what you would do with it instead.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

A team has one `AppContext` providing `{ user, theme, locale, cart, dispatch }`. The app has
about two hundred components and roughly sixty of them read the context. Adding one item to the
cart feels slow.

Answer in comments:

1. Explain, mechanically, why adding a cart item makes components that only read `theme`
   re-render.
2. Before changing anything, what would you measure, and with which tool?
3. Give two structural changes that would help, in the order you would try them — neither of
   which is memoisation.
4. A colleague proposes moving everything into Redux instead. Using this lesson, say what that
   would and would not fix, and what question you would ask first.

In [ ]:
// Your code here